In [22]:
! pip install Pillow


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np
from scipy import signal
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from matplotlib.ticker import FormatStrFormatter
import math
from PIL import Image, ImageDraw

## グリッドの定義

### 入力
- フロアマップの幅[px] : 2,837px ` map_width_px (int)`
- フロアマップの高さ[px]:  3,742px `map_height_px (int)`
- グリッドの一辺の長さ[px] : 30px `grid_size_px (int)`

### 出力
- 横方向のグリッド数 95 `num_grids_x`
- 縦方向のグリッド数 125 `num_grids_y`

### アルゴリズム
フロアマップの寸法とグリッドサイズから、横方向と縦方向にいくつのグリッドができるかを計算
- 横方向のグリッド数： $N_x = \lceil W / G \rceil = \lceil 2837 / 30 \rceil = \lceil 94.56... \rceil = 95$
- 縦方向のグリッド数： $N_y = \lceil H / G \rceil = \lceil 3742 / 30 \rceil = \lceil 124.73... \rceil = 125$



In [28]:
# 1. グリッドの定義
# 提供されたフロアマップの寸法を使用
map_width = 2837
map_height = 3742
# 提案されたグリッドサイズ
grid_size = 30 


num_grids_x = math.ceil(map_width / grid_size)
num_grids_y = math.ceil(map_height / grid_size)

img = Image.open("../../docs/floorMap/building＝14,floor=5_freeSpace.png")
draw = ImageDraw.Draw(img)
width, height = img.size
    
    # 縦線を描画
for x in range(0, width, grid_size):
    draw.line([(x, 0), (x, height)], fill="black")

    # 横線を描画
for y in range(0, height, grid_size):
     draw.line([(0, y), (width, y)], fill="black")

img.save("output/gridFloorMap.png")

print(f"--- グリッド定義 ---")
print(f"フロアマップの幅: {map_width} px")
print(f"フロアマップの高さ: {map_height} px")
print(f"グリッドサイズ: {grid_size} px")
print(f"横方向のグリッド数 (Nx): {num_grids_x}")
print(f"縦方向のグリッド数 (Ny): {num_grids_y}")
print("-" * 20)


--- グリッド定義 ---
フロアマップの幅: 2837 px
フロアマップの高さ: 3742 px
グリッドサイズ: 30 px
横方向のグリッド数 (Nx): 95
縦方向のグリッド数 (Ny): 125
--------------------


## 座標マッピング関数
歩行者の (x, y) 座標をグリッドの行インデックス、列インデックス、およびユニークなセルIDにマッピングします。


### 入力
- 歩行者のx座標:150? `px`
- 歩行者のy座標:200? `py`
- グリッドサイズ : 30`grid_size_px`
- 横方向のグリッド数 : 95 `num_grids_x`
- 横方向のグリッド数 : 125`num_grids_y`

### 出力
 (6, 5, '6_5')


### アルゴリズム
- row = floor(200 / 30) = floor(6.66...) = 6
- col = floor(150 / 30) = floor(5) = 5
- cell_id = "6_5"

- `math.floor()` を使用して、与えられた座標がどのグリッドの行と列に属するかを計算します。
-  計算された行と列から、`'row_col'`形式のユニークなセルIDを生成します。
- 座標がマップの有効範囲外である場合は `(None, None, None)` を返します。


In [32]:
px = 150
py = 200
row = math.floor(py / grid_size)
col = math.floor(px / grid_size)

# セルIDの生成 (例: 'row_col'形式)
cell_id = f"{row}_{col}"
print("`map_coordinates_to_cell_id` 関数を定義しました。")

`map_coordinates_to_cell_id` 関数を定義しました。


## 使用例

In [ ]:
# 1. グリッドの定義
# 提供されたフロアマップの寸法を使用
map_width = 2837
map_height = 3742
grid_size = 30 # 提案されたグリッドサイズ

Nx, Ny = define_grid(map_width, map_height, grid_size)

# 2. 座標のマッピング（セルIDへの変換）
print("\n--- 座標のマッピング例 ---")

# マップ内の有効な座標
test_px1, test_py1 = 150, 200
row1, col1, cell_id1 = map_coordinates_to_cell_id(test_px1, test_py1, grid_size, Nx, Ny)
if cell_id1:
    print(f"座標 ({test_px1}, {test_py1}) -> 行: {row1}, 列: {col1}, セルID: {cell_id1}")
else:
    print(f"座標 ({test_px1}, {test_py1}) -> マップの範囲外")

# マップの右下の端に近い座標
test_px2, test_py2 = 2836, 3741
row2, col2, cell_id2 = map_coordinates_to_cell_id(test_px2, test_py2, grid_size, Nx, Ny)
if cell_id2:
    print(f"座標 ({test_px2}, {test_py2}) -> 行: {row2}, 列: {col2}, セルID: {cell_id2}")
else:
    print(f"座標 ({test_px2}, {test_py2}) -> マップの範囲外")

# マップの範囲外の座標 (負の値)
test_px3, test_py3 = -10, 50
row3, col3, cell_id3 = map_coordinates_to_cell_id(test_px3, test_py3, grid_size, Nx, Ny)
if not cell_id3:
    print(f"座標 ({test_px3}, {test_py3}) -> マップの範囲外")

# マップの範囲外の座標 (大きすぎる値)
test_px4, test_py4 = 3000, 100
row4, col4, cell_id4 = map_coordinates_to_cell_id(test_px4, test_py4, grid_size, Nx, Ny)
if not cell_id4:
    print(f"座標 ({test_px4}, {test_py4}) -> マップの範囲外")

--- グリッド定義 ---
フロアマップの幅: 2837 px
フロアマップの高さ: 3742 px
グリッドサイズ: 30 px
横方向のグリッド数 (Nx): 95
縦方向のグリッド数 (Ny): 125
--------------------

--- 座標のマッピング例 ---
座標 (150, 200) -> 行: 6, 列: 5, セルID: 6_5
座標 (2836, 3741) -> 行: 124, 列: 94, セルID: 124_94
